# Kaggle Inference Only (CPU)

> **이 노트북의 역할**: 학습은 0회. 학습된 모델과 fit 결과를 Kaggle Dataset 으로 받아서 **test 이미지에 추론 + override 적용 → submission.csv** 만 생성.

## 노트북 구조

| Cell | 역할 | 시간 측정 |
|---|---|:-:|
| 1 | `pip install openvino + onnxruntime + ...` (Q&A #7 허용) | 외부 |
| 2 | imports, paths 자동 감지 (Kaggle / Local) | 외부 |
| 3 | SEED 고정 + 환경 transparency print | 외부 |
| 4 | OV/ORT session load (BETA/BIDIR via OV, ConvNeXt via ORT INT8) | 외부 |
| 5 | thresholds.json + AH12 pickle load | 외부 |
| 6 | **Test preload** (PIL + Pool 병렬, Q&A #5 허용) | 외부 |
| 7 | **Helper 함수 정의** (13D feature, softmax, AH12 helper, lazy cache) | 외부 |
| **8** | ⏱️ **시간 측정: ONNX forward + override rule 적용** | **내부** |
| 9 | submission.csv + md5 출력 | 외부 |

## 가속 전략

1. **OpenVINO Runtime CPU** — BETA/BIDIR ORT FP32 대비 2.94~3.25× 가속
2. **ORT INT8 (ConvNeXt)** — FP32 대비 1.68× 가속, OV Intel CPU plugin 미지원 우회
3. **Preload + PIL image read** (학습과 동일 resize 경로 — 재현성 보장)
4. **DataLoader 제거 + numpy batch slicing** — Q&A #6 허용
5. **multiprocessing.Pool** image read 병렬화 — Q&A #4 허용
6. **Lazy ConvNeXt forward** — R3/R4 candidate 만 추론 (전체 대비 ~7%)
7. **AH12 r1 pre-filter** — fft+coh 만 먼저, cc/lbp 는 통과 sample 만

⚠️ **CPU 전용**. Kaggle CPU 환경 (`Accelerator: None`) 기준.

## [Cell 1] 라이브러리 설치 (시간 측정 외부)

In [20]:
# 필요한 라이브러리 설치
# - openvino: ⭐ OpenVINO Runtime CPU 추론 (2026-05-15 migration)
#     OV FP32 가 ORT FP32 대비 2.94~3.25× 가속 (BETA/BIDIR)
#     양자화 없이 framework 변경만으로 가속 달성
# - onnxruntime: ConvNeXt-T INT8 추론 (OV Intel CPU plugin 미지원 우회)
# - opencv-python-headless: 13D feature 추출 (gray변환, Sobel, CC 등)
# - scikit-image: AH12 의 LBP feature
# - ⭐ numpy==2.4.1 + scikit-learn==1.8.0 + pandas==3.0.0 (pickle 호환성)
%pip install -q --upgrade \
    "numpy==2.4.1" \
    "scikit-learn==1.8.0" \
    "pandas==3.0.0" \
    openvino \
    onnxruntime \
    opencv-python-headless \
    scikit-image \
    2>&1 | tail -5
# ============================================================================
# ⭐ Kernel 자동 restart — 이미 import 된 옛 numpy/sklearn/pandas 강제로 새 버전
# ============================================================================
TARGET_NUMPY = '2.4.1'
TARGET_SKLEARN = '1.8.0'
TARGET_PANDAS = '3.0.0'

import importlib
try:
    np_mod = importlib.import_module('numpy')
    skl_mod = importlib.import_module('sklearn')
    pd_mod = importlib.import_module('pandas')
    cur_numpy = np_mod.__version__
    cur_sklearn = skl_mod.__version__
    cur_pandas = pd_mod.__version__
except Exception:
    cur_numpy = cur_sklearn = cur_pandas = 'unknown'

print()
print("━" * 60)
print(f"  현재 import 된 numpy   : {cur_numpy} (목표: {TARGET_NUMPY})")
print(f"  현재 import 된 sklearn : {cur_sklearn} (목표: {TARGET_SKLEARN})")
print(f"  현재 import 된 pandas  : {cur_pandas} (목표: {TARGET_PANDAS})")
print("━" * 60)

if cur_numpy != TARGET_NUMPY or cur_sklearn != TARGET_SKLEARN or cur_pandas != TARGET_PANDAS:
    print("⏎ 버전 불일치 → Kernel 자동 restart 진행...")
    print("   Restart 완료되면 'Run All' 다시 한 번 눌러주세요.")
    print("━" * 60)
    try:
        import IPython
        IPython.Application.instance().kernel.do_shutdown(restart=True)
    except Exception as e:
        print(f"⚠️ 자동 restart 실패 ({type(e).__name__}: {e})")
        print(f"   수동으로: Kaggle 메뉴 → Run → 'Restart kernel & Run All'")
else:
    print(f"✅ 버전 일치 완료 — 다음 cell 진행 OK")
    print("━" * 60)


Note: you may need to restart the kernel to use updated packages.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  현재 import 된 numpy   : 2.4.1 (목표: 2.4.1)
  현재 import 된 sklearn : 1.8.0 (목표: 1.8.0)
  현재 import 된 pandas  : 3.0.0 (목표: 3.0.0)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ 버전 일치 완료 — 다음 cell 진행 OK
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [23]:
! ls /kaggle/input/

competitions  datasets


## [Cell 2] Imports + 경로 자동 감지

Kaggle / Local 환경 자동 감지. Kaggle 에서는 `/kaggle/input/<dataset-name>/` 에서 artifact 로드.

In [24]:
import os
import time
import json
import pickle
import hashlib
import platform
import subprocess
from pathlib import Path
from multiprocessing import Pool

import numpy as np
import pandas as pd
import cv2
import onnxruntime as ort
import openvino as ov   # ⭐ OpenVINO Runtime (2026-05-15 migration)
from skimage.feature import local_binary_pattern

# ============================================================================
# 경로 — Kaggle 환경에 맞춰 명시적 hardcode
# ============================================================================
if Path('/kaggle/input').exists():
    ARTIFACTS_DIR = Path('/kaggle/input/datasets/leesangyub2/onnx-pkl')
    TEST_DIR = Path('/kaggle/input/competitions/2026-1-machine-learning-in-prace/test/test/images')
    OUT_DIR = Path('/kaggle/working')
    IS_KAGGLE = True
else:
    try:
        _HERE = Path(__file__).resolve().parent
    except NameError:
        _vsc = globals().get('__vsc_ipynb_file__')
        _HERE = Path(_vsc).resolve().parent if _vsc else Path.cwd().resolve()
    ARTIFACTS_DIR = _HERE / 'kaggle_artifacts'
    for p in [_HERE, *_HERE.parents]:
        if (p / 'competition_dataset' / 'NEU-DET_open').is_dir():
            TEST_DIR = p / 'competition_dataset' / 'NEU-DET_open' / 'test' / 'images'
            break
    else:
        TEST_DIR = _HERE / 'test'
    OUT_DIR = _HERE
    IS_KAGGLE = False

OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 ARTIFACTS_DIR : {ARTIFACTS_DIR}")
print(f"📂 TEST_DIR      : {TEST_DIR}")
print(f"📂 OUT_DIR       : {OUT_DIR}")
print(f"🌐 IS_KAGGLE     : {IS_KAGGLE}")

assert ARTIFACTS_DIR.exists(), f"ARTIFACTS_DIR 없음: {ARTIFACTS_DIR}"
assert TEST_DIR.exists(), f"TEST_DIR 없음: {TEST_DIR}"
for fn in ['student_BETA-LION.onnx', 'student_BIDIR.onnx',
           'teacher_convnext_tiny.onnx', 'thresholds.json', 'ah12_state.pkl']:
    assert (ARTIFACTS_DIR / fn).exists(), f"Artifact 없음: {fn}"
print(f"✅ 모든 artifact 발견")


📂 ARTIFACTS_DIR : /kaggle/input/datasets/leesangyub2/onnx-pkl
📂 TEST_DIR      : /kaggle/input/competitions/2026-1-machine-learning-in-prace/test/test/images
📂 OUT_DIR       : /kaggle/working
🌐 IS_KAGGLE     : True
✅ 모든 artifact 발견


## [Cell 3] SEED 고정 + 환경 정보 print (CLAUDE.md 0.4.3)

검수자가 어떤 환경에서 추론된 결과인지 즉시 확인할 수 있도록 명시.

In [25]:
import random
import sklearn   # ⭐ pickle 호환성 위해 명시 import
import skimage   # ⭐ 13D feature 의 LBP 라이브러리

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print("━" * 60)
print("  🔒 RNG seed configuration")
print("━" * 60)
print(f"  SEED                : {SEED}")
print(f"  PYTHONHASHSEED      : {os.environ.get('PYTHONHASHSEED')}")
print(f"  random.seed         : {SEED}")
print(f"  np.random.seed      : {SEED}")
print("━" * 60)
print("  🖥️  Environment")
print("━" * 60)
print(f"  OS                  : {platform.platform()}")
print(f"  Python              : {platform.python_version()}")
print(f"  CPU arch            : {platform.machine()}")
print(f"  CPU count           : {os.cpu_count()}")
print(f"  numpy               : {np.__version__}")
print(f"  pandas              : {pd.__version__}")
print(f"  scikit-learn        : {sklearn.__version__}")
print(f"  scikit-image        : {skimage.__version__}")
print(f"  onnxruntime         : {ort.__version__}")
print(f"  openvino            : {ov.__version__}")   # ⭐ OV version
print(f"  opencv              : {cv2.__version__}")
print("━" * 60)

TARGET = {'numpy': '2.4.1', 'scikit-learn': '1.8.0'}
mismatches = []
if np.__version__ != TARGET['numpy']:
    mismatches.append(f"numpy: {np.__version__} (목표 {TARGET['numpy']})")
if sklearn.__version__ != TARGET['scikit-learn']:
    mismatches.append(f"sklearn: {sklearn.__version__} (목표 {TARGET['scikit-learn']})")
if mismatches:
    print("⚠️ 버전 불일치 — Cell 1 의 pip install + Kernel Restart 권장:")
    for m in mismatches:
        print(f"   {m}")
else:
    print("✅ numpy / sklearn 버전 일치 — pickle 호환 OK")
print("━" * 60)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🔒 RNG seed configuration
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  SEED                : 42
  PYTHONHASHSEED      : 42
  random.seed         : 42
  np.random.seed      : 42
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🖥️  Environment
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  OS                  : Linux-6.6.122+-x86_64-with-glibc2.35
  Python              : 3.12.12
  CPU arch            : x86_64
  CPU count           : 4
  numpy               : 2.4.1
  pandas              : 3.0.0
  scikit-learn        : 1.8.0
  scikit-image        : 0.26.0
  onnxruntime         : 1.26.0
  openvino            : 2026.1.0-21367-63e31528c62-releases/2026/1
  opencv              : 4.13.0
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ numpy / sklearn 버전 일치 — pickle 호환 OK
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


## [Cell 4] ONNX / OV 모델 로드 (시간 측정 외부)

- **OV** (OpenVINO Runtime): BETA/BIDIR (MobileNetV3-Small) → ORT FP32 대비 2.94~3.25× 가속
- **ORT** (ONNX Runtime): ConvNeXt-T INT8 → OV Intel CPU plugin `SequenceMark dynamic rank` 오류로 우회
- `intra_op_num_threads = CPU 코어 수` → 모든 코어 활용
- `ORT_ENABLE_ALL` 그래프 최적화 → BN-Conv fusion 등 자동 적용

In [26]:
import os
print("ARTIFACTS_DIR 안의 모든 파일:")
for f in sorted(os.listdir(ARTIFACTS_DIR)):
    size_mb = os.path.getsize(ARTIFACTS_DIR / f) / 1024**2
    print(f"  {f}  ({size_mb:.1f} MB)")

ARTIFACTS_DIR 안의 모든 파일:
  ah12_state.pkl  (0.0 MB)
  student_BETA-LION.onnx  (5.9 MB)
  student_BETA-LION_SHAVED.onnx  (6.1 MB)
  student_BETA-LION_int8.onnx  (1.6 MB)
  student_BIDIR.onnx  (5.9 MB)
  student_BIDIR_SHAVED.onnx  (6.1 MB)
  student_BIDIR_int8.onnx  (1.6 MB)
  teacher_convnext_tiny.onnx  (106.2 MB)
  teacher_convnext_tiny_int8.onnx  (27.2 MB)
  teacher_convnext_tiny_int8.onnx.V12_BACKUP  (27.2 MB)
  teacher_convnext_tiny_ov_fp32.bin  (53.1 MB)
  teacher_convnext_tiny_ov_fp32.xml  (0.4 MB)
  teacher_convnext_tiny_ov_int8.bin  (26.9 MB)
  teacher_convnext_tiny_ov_int8.xml  (0.6 MB)
  thresholds.json  (0.0 MB)


In [27]:
# ============================================================================
# ⭐ Mixed runtime (2026-05-15) — OV for BETA/BIDIR, ORT for ConvNeXt
#
#    원인: ConvNeXt-T (timm) ONNX 는 OV Intel CPU plugin 에서
#          `SequenceMark dynamic rank` 에러로 compile 실패.
#          (timm 의 LayerNorm/DropPath path 가 ONNX 변환 시 그래프 내부에
#           rank 추론 안 되는 텐서 흔적을 남김. Apple Silicon ARM plugin 은
#           통과하지만 Intel CPU plugin 은 strict 검사로 reject.)
#
#    해결: 큰 가속 받는 BETA/BIDIR (MobileNetV3) 만 OV 로 이행.
#          ConvNeXt 는 ORT 그대로 (V12 INT8 또는 FP32).
#
#    측정값:
#      - BETA  ORT 68.9ms → OV 23.4ms  (2.94×) ⭐
#      - BIDIR ORT 78.2ms → OV 24.1ms  (3.25×) ⭐
#      - ConvNeXt ORT V12 INT8 ≈ 155ms  (OV blocked)
# ============================================================================
IMG_H, IMG_W = 192, 192

# ── OV: BETA + BIDIR (MobileNetV3-Small) ────────────────────────────────────
core = ov.Core()


def _load_ov_compiled(onnx_path, label):
    """OV compile with explicit dynamic-batch shape — Intel CPU plugin 호환."""
    model = core.read_model(str(onnx_path))
    input_node = model.input(0)
    try:
        model.reshape({input_node.any_name: ov.PartialShape([-1, 3, IMG_H, IMG_W])})
    except Exception as e:
        print(f"  ⚠️ {label}: reshape 실패 ({e}) — fallback to original shape")
    return core.compile_model(model, device_name='CPU')


# ── ORT: ConvNeXt-T (timm 의 SequenceMark 문제로 OV 안 됨) ──────────────────
USE_INT8_CNXT = True   # V12 static QDQ INT8 (FP32 260ms vs INT8 155ms — 1.68× 가속)

sess_opts = ort.SessionOptions()
sess_opts.intra_op_num_threads = max(1, os.cpu_count() or 4)
sess_opts.inter_op_num_threads = 1
sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess_opts.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL

# ── Paths ───────────────────────────────────────────────────────────────────
beta_name  = 'student_BETA-LION.onnx'
bidir_name = 'student_BIDIR.onnx'
cnxt_name  = 'teacher_convnext_tiny_int8.onnx' if USE_INT8_CNXT else 'teacher_convnext_tiny.onnx'

beta_path  = ARTIFACTS_DIR / beta_name
bidir_path = ARTIFACTS_DIR / bidir_name
cnxt_path  = ARTIFACTS_DIR / cnxt_name

# ── Load ────────────────────────────────────────────────────────────────────
t0 = time.perf_counter()
compiled_beta  = _load_ov_compiled(beta_path,  'BETA')
compiled_bidir = _load_ov_compiled(bidir_path, 'BIDIR')
sess_cnxt      = ort.InferenceSession(str(cnxt_path),
                                       sess_options=sess_opts,
                                       providers=['CPUExecutionProvider'])
INAME_CNXT     = sess_cnxt.get_inputs()[0].name
load_t = time.perf_counter() - t0

md5_beta  = hashlib.md5(beta_path.read_bytes()).hexdigest()
md5_bidir = hashlib.md5(bidir_path.read_bytes()).hexdigest()
md5_cnxt  = hashlib.md5(cnxt_path.read_bytes()).hexdigest()

# OV output 키 캐시 (forward 마다 dict lookup 절감)
OUT_BETA  = compiled_beta.output(0)
OUT_BIDIR = compiled_bidir.output(0)

print(f"✅ Loaded mixed runtime in {load_t:.3f}s")
print(f"  OV  Available devices: {core.available_devices}")
try:
    print(f"  OV  CPU type         : {core.get_property('CPU', 'FULL_DEVICE_NAME')}")
except Exception:
    pass
print(f"  OV  Input shape       : [N, 3, {IMG_H}, {IMG_W}]  (batch dynamic)")
print(f"  OV  BETA   : {beta_name:35s} md5={md5_beta[:16]}..")
print(f"  OV  BIDIR  : {bidir_name:35s} md5={md5_bidir[:16]}..")
print(f"  ORT CnxT   : {cnxt_name:35s} md5={md5_cnxt[:16]}..  "
      f"({'INT8 V12' if USE_INT8_CNXT else 'FP32'})")


✅ Loaded mixed runtime in 1.185s
  OV  Available devices: ['CPU']
  OV  CPU type         :            Intel(R) Xeon(R) CPU @ 2.20GHz
  OV  Input shape       : [N, 3, 192, 192]  (batch dynamic)
  OV  BETA   : student_BETA-LION.onnx              md5=33bcec9126d0650e..
  OV  BIDIR  : student_BIDIR.onnx                  md5=57b96255e1360547..
  ORT CnxT   : teacher_convnext_tiny_int8.onnx     md5=9824fddd500f0607..  (INT8 V12)


## [Cell 5] thresholds.json + AH12 sklearn 객체 load

- `thresholds.json` : R3 / R4 / P_VETO / BIDIR_ROL_VETO + wA + class_bias
- `ah12_state.pkl` : sklearn `scaler`, `clf` + Gaussian fit params

In [28]:
with open(ARTIFACTS_DIR / 'thresholds.json') as f:
    TH = json.load(f)

with open(ARTIFACTS_DIR / 'ah12_state.pkl', 'rb') as f:
    AH12 = pickle.load(f)

# Constants
CRA, INC, PAT, PIT, ROL, SCR = 0, 1, 2, 3, 4, 5
class_names = TH['class_names']
IMG_SIZE = int(TH['IMG_SIZE'])
NUM_CLASSES = int(TH['NUM_CLASSES'])

# Champion ensemble anchor
wA = float(TH['wA'])
class_bias = np.array(TH['class_bias'], dtype=np.float32)

# Override 룰 임계값
ALLOWED_TARGETS = [(int(t[0]), float(t[1])) for t in TH['ALLOWED_TARGETS']]
ALLOWED_MAP = {t: cf for (t, cf) in ALLOWED_TARGETS}
P_VETO = float(TH['P_VETO'])
PIT_CNXT_FLOOR    = float(TH['R4_cnxt_floor'])
PIT_BETA_FLOOR    = float(TH['R4_beta_pit_floor'])
PIT_PFLOOR        = float(TH['R4_p_pit_floor'])
BIDIR_ROL_VETO    = float(TH['BIDIR_ROL_VETO'])
RULE_3WAY_VALID   = bool(TH['rule_3way_valid'])
R4_VALID          = bool(TH['r4_valid'])
AH12_VALID        = bool(TH['ah12_valid'])

print(f"✅ Loaded thresholds + AH12 state")
print(f"  wA={wA}, class_bias={class_bias.tolist()}")
print(f"  ALLOWED_TARGETS = {[(class_names[t], cf) for t, cf in ALLOWED_TARGETS]}")
print(f"  P_VETO = {P_VETO:.4f}")
print(f"  R4: cnxt_floor={PIT_CNXT_FLOOR}, beta_pit_floor={PIT_BETA_FLOOR}, "
      f"p_pit_floor={PIT_PFLOOR}, BIDIR_ROL_VETO={BIDIR_ROL_VETO}")
print(f"  Valid flags: R3={RULE_3WAY_VALID}, R4={R4_VALID}, AH12={AH12_VALID}")

✅ Loaded thresholds + AH12 state
  wA=0.7, class_bias=[2.0, -1.100000023841858, 1.899999976158142, -0.10000000149011612, 1.2000000476837158, -0.699999988079071]
  ALLOWED_TARGETS = [('inclusion', 0.99), ('scratches', 0.0)]
  P_VETO = 0.3789
  R4: cnxt_floor=0.0, beta_pit_floor=0.99, p_pit_floor=0.0, BIDIR_ROL_VETO=0.9476
  Valid flags: R3=True, R4=True, AH12=True


## [Cell 6] Test 이미지 Preload (시간 측정 외부, Q&A #5 허용)

- PIL.Image (학습과 동일 resize 경로 — 재현성 보장)
- `multiprocessing.Pool(4)` 병렬 read

> Q&A #5 허용: test data 를 transform 한 상태로 미리 메모리에 로드 가능.

In [29]:
# Top-level 함수 (multiprocessing pickle 가능하도록)
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)
_IMG_SIZE = IMG_SIZE
_TEST_DIR = TEST_DIR


def _load_one(fn):
    """한 이미지의 (model_input, resized_rgb_uint8) 반환.
    resized_rgb 는 13D feature 에도 재사용.

    ⭐ V9-HONEST .pth 학습/추론과 동일 — PIL.Image.open + PIL.BILINEAR resize.
       (이전 cv2 경로는 JPEG 디코더 + bilinear kernel 이 미세하게 달라
        boundary sample 의 argmax 가 뒤집힘. test_018 = 4 → 0 회복 검증됨.)
    """
    # 🔒 worker process (fork/spawn 모두) 에서도 안전하게 — 함수 내부 import
    from PIL import Image
    pil = Image.open(_TEST_DIR / fn).convert('RGB').resize(
        (_IMG_SIZE, _IMG_SIZE), Image.BILINEAR)
    img_resized = np.array(pil)  # uint8 RGB, PIL-resized (compute_feat13 source)
    # Model input
    arr = img_resized.astype(np.float32).transpose(2, 0, 1) / 255.0
    arr = (arr - MEAN) / STD
    return arr.astype(np.float32), img_resized


# Test 파일 sorted (재현성)
test_files = sorted([f for f in os.listdir(TEST_DIR)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
N = len(test_files)
print(f"Test images: {N}")

# 병렬 preload
N_WORKERS = max(1, min(4, os.cpu_count() or 4))
t_preload = time.perf_counter()
with Pool(N_WORKERS) as p:
    results = p.map(_load_one, test_files)
t_preload = time.perf_counter() - t_preload

X = np.stack([r[0] for r in results])          # (N, 3, 192, 192) float32 normalized
resized_imgs = [r[1] for r in results]          # list of (192, 192, 3) uint8 RGB

print(f"Preload: {t_preload:.3f}s ({t_preload*1000/N:.2f} ms/img, "
      f"workers={N_WORKERS}), X.shape={X.shape}")

Test images: 180
Preload: 0.626s (3.48 ms/img, workers=4), X.shape=(180, 3, 192, 192)


## [Cell 7] Helper 함수 정의 (timing window 밖)

- 13D feature 4 helper (`fft_power_ratio` / `coherence` / `cc_count` / `lbp_hist`) + `compute_feat13`
- `softmax_np`, AH12 Gaussian helper (`_ll_ah12`, `_dmaha_ah12`)
- Lazy cache (`get_feat_r1` / `get_feat13` / `get_p_pit`) — Cell 8 에서 호출

In [30]:
# ============================================================================
# 함수 정의 전용 cell (timing window 밖)
#   - 13D feature 4 helper (fft / coh / cc / lbp) + compute_feat13
#   - softmax_np
#   - lazy cache (get_feat_r1 / get_feat13 / get_p_pit) + cache dict
#   - AH12 helper (_ll / _dmaha)
#   - timing 측정 list (_feat_r1_compute_time 등)
#
# Cell 20 은 cache.clear() + timing + inference 만 — 함수 정의 0.
# ============================================================================
LBP_P, LBP_R, LBP_BINS = 8, 1, 10


def fft_power_ratio(gray_f32):
    F_ = np.fft.fftshift(np.fft.fft2(gray_f32)); mag = np.abs(F_)
    H, W = gray_f32.shape; cy, cx = H // 2, W // 2
    yy, xx = np.indices((H, W))
    r = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
    r_n = r / r.max()
    return float(mag[(r_n >= 0.4) & (r_n <= 1.0)].sum() / (mag.sum() + 1e-8))


def coherence(gray_f32):
    gx = cv2.Sobel(gray_f32, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray_f32, cv2.CV_32F, 0, 1, ksize=3)
    Sxx = cv2.GaussianBlur(gx * gx, (5, 5), 1.0)
    Sxy = cv2.GaussianBlur(gx * gy, (5, 5), 1.0)
    Syy = cv2.GaussianBlur(gy * gy, (5, 5), 1.0)
    tr = Sxx + Syy
    det = Sxx * Syy - Sxy * Sxy
    sq = np.sqrt(np.maximum(tr * tr / 4 - det, 0))
    return float(((tr / 2 + sq - (tr / 2 - sq)) /
                  (tr / 2 + sq + tr / 2 - sq + 1e-8)).mean())


def cc_count(gray_u8):
    _, th = cv2.threshold(gray_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    num, _ = cv2.connectedComponents(th)
    return int(num)


def lbp_hist(gray_u8):
    lbp = local_binary_pattern(gray_u8, LBP_P, LBP_R, method='uniform')
    h, _ = np.histogram(lbp.ravel(), bins=LBP_BINS, range=(0, LBP_BINS), density=True)
    return h.astype(np.float32)


def compute_feat13(rgb_u8):
    gray_u8 = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2GRAY)
    gray_f32 = gray_u8.astype(np.float32) / 255.0
    return np.concatenate([
        [fft_power_ratio(gray_f32),
         coherence(gray_f32),
         float(cc_count(gray_u8))],
        lbp_hist(gray_u8),
    ])


# ─── softmax (numpy, inference 전용) ─────────────────────────────────────
def softmax_np(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)


# ─── AH12 Gaussian helper (Cell 20 에서 호출, 정의는 여기서 한 번) ──────
def _ll_ah12(x, mu, Sinv, ld):
    return -0.5 * ((x - mu) @ Sinv @ (x - mu)) - 0.5 * ld

def _dmaha_ah12(x, mu, Sinv):
    return float((x - mu) @ Sinv @ (x - mu))


# ─── Lazy cache (timing 안에서 lazy 호출, dict 는 여기서 만들고 cell 20 에서 clear) ──
_feat_r1_cache = {}
_feat_cache    = {}
_ppit_cache    = {}
_feat_r1_compute_time = [0.0]
_feat13_compute_time  = [0.0]
_predict_proba_time   = [0.0]


def get_feat_r1(i):
    """fft + coh 만 — AH12 r1 빠른 검사."""
    if i in _feat_cache:
        return _feat_cache[i][:2]
    if i not in _feat_r1_cache:
        _t0 = time.perf_counter()
        gray_u8 = cv2.cvtColor(resized_imgs[i], cv2.COLOR_RGB2GRAY)
        gray_f32 = gray_u8.astype(np.float32) / 255.0
        _feat_r1_cache[i] = np.array(
            [fft_power_ratio(gray_f32), coherence(gray_f32)])
        _feat_r1_compute_time[0] += time.perf_counter() - _t0
    return _feat_r1_cache[i]


def get_feat13(i):
    """full 13D — r1 통과 또는 R3/R4 의 p_pit 필요할 때."""
    if i not in _feat_cache:
        _t0 = time.perf_counter()
        if i in _feat_r1_cache:
            # fft+coh 재사용 — cc+lbp 만 추가
            gray_u8 = cv2.cvtColor(resized_imgs[i], cv2.COLOR_RGB2GRAY)
            _feat_cache[i] = np.concatenate([
                _feat_r1_cache[i],
                [float(cc_count(gray_u8))],
                lbp_hist(gray_u8),
            ])
        else:
            _feat_cache[i] = compute_feat13(resized_imgs[i])
        _feat13_compute_time[0] += time.perf_counter() - _t0
    return _feat_cache[i]


def get_p_pit(i):
    if i not in _ppit_cache:
        f = get_feat13(i)
        _t0 = time.perf_counter()
        _ppit_cache[i] = float(AH12['clf'].predict_proba(
            AH12['scaler'].transform(f[None, :]))[0, 1])
        _predict_proba_time[0] += time.perf_counter() - _t0
    return _ppit_cache[i]


print("✅ 함수 정의 완료")

✅ 함수 정의 완료


## [Cell 8] ⏱️ 시간 측정 구간 — ONNX Forward + Override 적용

측정 구간 안에서 하는 일:
1. BETA/BIDIR forward (OV) — 전체 180장
2. Softmax + ensemble (champion = 0.7·BETA + 0.3·BIDIR + class_bias)
3. ConvNeXt lazy forward (R3/R4 candidate 만 — ~7%)
4. Override stack (R3 → R4 → AH12) + AH12 r1 pre-filter

In [31]:
# ── Warmup (timed window 밖) ─────────────────────────────────
_ = compiled_beta([X])
_ = compiled_bidir([X])
_ = sess_cnxt.run(None, {INAME_CNXT: X[0:12]})

# ── Lazy cache reset (timing window 밖) ──────────────────────
_feat_r1_cache.clear()
_feat_cache.clear()
_ppit_cache.clear()
_feat_r1_compute_time[0] = 0.0
_feat13_compute_time[0]  = 0.0
_predict_proba_time[0]   = 0.0
t_stages = {}
print("✅ Warmup + cache reset done — inference 시작")

# ═════════════════════════════════════════════════════════════
# ⏱️ TIMING START — 오직 inference 만
# ═════════════════════════════════════════════════════════════
t0_inf = time.perf_counter()

# ─── (1) BETA + BIDIR forward (OV) ──────────────────────────────────────
_t = time.perf_counter()
lA = compiled_beta([X])[OUT_BETA]
lB = compiled_bidir([X])[OUT_BIDIR]
t_stages['1_beta_bidir_forward (OV)'] = time.perf_counter() - _t

# ─── (2) Champion ensemble ──────────────────────────────────────────────
_t = time.perf_counter()
probA = softmax_np(lA); probB = softmax_np(lB)
champ_prob  = wA * probA + (1 - wA) * probB
champ_final = softmax_np(np.log(champ_prob + 1e-12) + class_bias[None, :])
champ_pred  = champ_final.argmax(1)
beta_argmax = probA.argmax(1)
t_stages['2_ensemble (numpy)'] = time.perf_counter() - _t

# ─── (2b) ConvNeXt-T lazy forward — tighter filter ──────────────────────
_t = time.perf_counter()
r3_possible = np.array([int(ba) in ALLOWED_MAP for ba in beta_argmax])
r4_possible = (beta_argmax == PIT)
candidates  = np.where((champ_pred != beta_argmax) & (r3_possible | r4_possible))[0]
n_cand = len(candidates)
lC = np.zeros((N, NUM_CLASSES), dtype=np.float32)
if n_cand > 0:
    lC[candidates] = sess_cnxt.run(None, {INAME_CNXT: X[candidates]})[0]
probC = softmax_np(lC)
cnxt_argmax = probC.argmax(1); cnxt_conf = probC.max(1)
t_stages[f'2b_cnxt_forward (n_cand={n_cand})'] = time.perf_counter() - _t

# ─── (3) Override stack — lazy 호출 ─────────────────────────────────────
final_pred = champ_pred.copy()
fired_log = []
veto_log = []

# Layer 1: R3_pertarget (with P_VETO) — get_p_pit lazy
_t = time.perf_counter()
if RULE_3WAY_VALID and ALLOWED_MAP:
    for i in range(N):
        if champ_pred[i] == beta_argmax[i]: continue
        if not (beta_argmax[i] == cnxt_argmax[i]
                and int(beta_argmax[i]) in ALLOWED_MAP
                and cnxt_conf[i] >= ALLOWED_MAP[int(beta_argmax[i])]):
            continue
        from_cls = int(champ_pred[i]); to_cls = int(beta_argmax[i])
        if from_cls == PIT and to_cls == INC:
            p_pit_i = get_p_pit(i)
            if p_pit_i >= P_VETO:
                veto_log.append({'id': test_files[i], 'rule': 'R3_pit_to_inc_veto',
                                  'p_pit': p_pit_i})
                continue
        else:
            p_pit_i = get_p_pit(i)
        final_pred[i] = to_cls
        fired_log.append({'id': test_files[i], 'rule': 'R3',
                           'from': from_cls, 'to': to_cls,
                           'p_pit': p_pit_i})
t_stages['3a_R3_layer (loop+lazy)'] = time.perf_counter() - _t

# Layer 2: R4_pit (with BIDIR_ROL_VETO) — get_p_pit lazy
_t = time.perf_counter()
if R4_VALID:
    beta_pit  = probA[:, PIT]
    cnxt_pit  = probC[:, PIT]
    bidir_rol = probB[:, ROL]
    fired_ids = {ov_log['id'] for ov_log in fired_log}
    for i in range(N):
        if test_files[i] in fired_ids: continue
        if champ_pred[i] == beta_argmax[i]: continue
        if not (beta_argmax[i] == cnxt_argmax[i]
                and beta_argmax[i] == PIT
                and cnxt_pit[i]  >= PIT_CNXT_FLOOR
                and beta_pit[i]  >= PIT_BETA_FLOOR):
            continue
        p_pit_i = get_p_pit(i)
        if p_pit_i < PIT_PFLOOR:
            continue
        if bidir_rol[i] >= BIDIR_ROL_VETO:
            veto_log.append({'id': test_files[i], 'rule': 'R4_bidir_rol_veto',
                              'bidir_rol': float(bidir_rol[i])})
            continue
        final_pred[i] = PIT
        fired_log.append({'id': test_files[i], 'rule': 'R4',
                           'from': int(champ_pred[i]), 'to': PIT,
                           'beta_pit': float(beta_pit[i]),
                           'cnxt_pit': float(cnxt_pit[i]),
                           'p_pit': p_pit_i})
t_stages['3b_R4_layer (loop+lazy)'] = time.perf_counter() - _t

# Layer 3: AH12 — r1 pre-filter (fft+coh) → r1 통과 시 full 13D
_t = time.perf_counter()
n_inc = 0; n_r1_pass = 0
if AH12_VALID:
    mu_inc3, Sinv_inc3, ld_inc3 = AH12['mu_inc3'], AH12['Sinv_inc3'], AH12['ld_inc3']
    mu_pit3, Sinv_pit3, ld_pit3 = AH12['mu_pit3'], AH12['Sinv_pit3'], AH12['ld_pit3']
    A_r1_fft = AH12['A_r1_fft']; A_r1_coh = AH12['A_r1_coh']
    A_r2_thr = AH12['A_r2_thr']; A_r3_thr = AH12['A_r3_thr']; A_r4_thr = AH12['A_r4_thr']

    fired_ids = {ov_log['id'] for ov_log in fired_log}
    for i in range(N):
        if test_files[i] in fired_ids: continue
        if int(final_pred[i]) != INC: continue
        n_inc += 1
        # Step 1: r1 만 먼저 (fft + coh, cc/lbp skip)
        x_r1 = get_feat_r1(i)
        if not (x_r1[0] < A_r1_fft and x_r1[1] > A_r1_coh):
            continue
        n_r1_pass += 1
        # Step 2: r1 pass → full 13D 로 upgrade (cc + lbp 추가)
        x = get_feat13(i)
        r2 = (_ll_ah12(x[:3], mu_pit3, Sinv_pit3, ld_pit3) -
              _ll_ah12(x[:3], mu_inc3, Sinv_inc3, ld_inc3)) > A_r2_thr
        if not r2: continue
        dp = _dmaha_ah12(x[:3], mu_pit3, Sinv_pit3)
        di = _dmaha_ah12(x[:3], mu_inc3, Sinv_inc3)
        r3 = (dp < di) and (dp < A_r3_thr)
        if not r3: continue
        p_pit_i = get_p_pit(i)
        if not (p_pit_i > A_r4_thr): continue
        final_pred[i] = PIT
        fired_log.append({'id': test_files[i], 'rule': 'AH12',
                           'from': INC, 'to': PIT,
                           'p_pit': p_pit_i})
t_stages[f'3c_AH12 (n_inc={n_inc}, r1_pass={n_r1_pass})'] = time.perf_counter() - _t

t_inf = time.perf_counter() - t0_inf
# ═════════════════════════════════════════════════════════════
# ⏱️ TIMING END
# ═════════════════════════════════════════════════════════════

# ─── Output (timing 밖) ─────────────────────────────────────────────────
print(f"\n{'═'*72}")
print(f"  ⏱️  TOTAL Inference: {t_inf:.4f}s  ({t_inf*1000/N:.2f} ms/img)")
print(f"{'═'*72}")

print(f"\n  📊 Stage wall time")
print(f"  {'-'*70}")
stage_sum = 0.0
for name, sec in t_stages.items():
    pct = sec / t_inf * 100
    bar = '█' * int(pct / 2)
    print(f"  {name:44s} {sec*1000:7.1f} ms  {pct:5.1f}%  {bar}")
    stage_sum += sec
unaccounted = t_inf - stage_sum
print(f"  {'(unaccounted: setup/print)':44s} {unaccounted*1000:7.1f} ms  "
      f"{unaccounted/t_inf*100:5.1f}%")

print(f"\n  🔍 Lazy cache cumulative (R3/R4/AH12 layer time 의 일부)")
print(f"  {'-'*70}")
print(f"  AH12 r1 (fft+coh) × {len(_feat_r1_cache):3d} sample      "
      f"{_feat_r1_compute_time[0]*1000:7.1f} ms  "
      f"({_feat_r1_compute_time[0]*1000/max(len(_feat_r1_cache),1):.2f} ms/img)")
print(f"  full 13D (cc+lbp) × {len(_feat_cache):3d} sample        "
      f"{_feat13_compute_time[0]*1000:7.1f} ms  "
      f"({_feat13_compute_time[0]*1000/max(len(_feat_cache),1):.2f} ms/img)")
print(f"  predict_proba × {len(_ppit_cache):3d} sample            "
      f"{_predict_proba_time[0]*1000:7.1f} ms  "
      f"({_predict_proba_time[0]*1000/max(len(_ppit_cache),1):.2f} ms/img)")

if n_inc > 0:
    r1_skip = n_inc - n_r1_pass
    avg_full = _feat13_compute_time[0] / max(len(_feat_cache), 1)
    print(f"\n  💡 AH12 r1 pre-filter 효과: {r1_skip}/{n_inc} sample 의 cc+lbp skip "
          f"≈ -{r1_skip * avg_full * 1000:.1f} ms 절감")

print(f"\n  Override fires: {len(fired_log)}")
for ov_log in fired_log:
    extra = ', '.join(f"{k}={v:.3f}" if isinstance(v, float) else f"{k}={v}"
                       for k, v in ov_log.items() if k not in ('id', 'rule', 'from', 'to'))
    print(f"    [{ov_log['rule']:5s}] {ov_log['id']}: {class_names[ov_log['from']]} → {class_names[ov_log['to']]}  {extra}")
print(f"\n  Veto blocks: {len(veto_log)}")
for v in veto_log:
    extra = ', '.join(f"{k}={v_:.3f}" if isinstance(v_, float) else f"{k}={v_}"
                       for k, v_ in v.items() if k not in ('id', 'rule'))
    print(f"    [{v['rule']}] {v['id']}: {extra}")

print(f"\n  Final prediction distribution:")
for c, name in enumerate(class_names):
    print(f"    {name:20s}: {int((final_pred == c).sum())}")


✅ Warmup + cache reset done — inference 시작

════════════════════════════════════════════════════════════════════════
  ⏱️  TOTAL Inference: 1.6123s  (8.96 ms/img)
════════════════════════════════════════════════════════════════════════

  📊 Stage wall time
  ----------------------------------------------------------------------
  1_beta_bidir_forward (OV)                      582.0 ms   36.1%  ██████████████████
  2_ensemble (numpy)                               0.7 ms    0.0%  
  2b_cnxt_forward (n_cand=12)                    770.5 ms   47.8%  ███████████████████████
  3a_R3_layer (loop+lazy)                         68.7 ms    4.3%  ██
  3b_R4_layer (loop+lazy)                         87.3 ms    5.4%  ██
  3c_AH12 (n_inc=29, r1_pass=3)                  102.7 ms    6.4%  ███
  (unaccounted: setup/print)                       0.5 ms    0.0%

  🔍 Lazy cache cumulative (R3/R4/AH12 layer time 의 일부)
  ----------------------------------------------------------------------
  AH12 r1 (fft+coh)

## [Cell 9] submission.csv 저장 + md5 + 예상 점수

CLAUDE.md Rule 10:
- `Id, Expected, inference_time_sec` 3 columns
- 모든 row 에 동일 `inference_time_sec` broadcast

In [32]:
#모든 row 에 inference_time_sec broadcast
submission = pd.DataFrame({
    'Id': test_files,
    'Expected': final_pred.astype(int),
    'inference_time_sec': round(float(t_inf), 4),
})
submission_path = OUT_DIR / 'submission.csv'
submission.to_csv(submission_path, index=False)
sub_md5 = hashlib.md5(submission_path.read_bytes()).hexdigest()

print("━" * 70)
print(f"  submission.csv saved → {submission_path}")
print(f"  rows                  : {len(submission)}")
print(f"  md5                   : {sub_md5}")
print(f"  inference_time_sec    : {round(float(t_inf), 4)}")
print(f"  override fires        : {len(fired_log)}")
print(f"  veto blocks           : {len(veto_log)}")
print("━" * 70)

# Final prediction distribution
print(f"\n  Final prediction distribution:")
for c, name in enumerate(class_names):
    print(f"    {name:20s}: {int((final_pred == c).sum())}")

# competition score (latency 기준)
penalty = max(0.0, (t_inf - 1.0) * 2.5)
print(f"\n🏆 T_inference : {t_inf:.4f}s")
print(f"   Latency 페널티 : -{penalty:.2f}")
if t_inf > 30:
    print(f"   ⚠️  실격 (T > 30s)")
elif t_inf > 1.0:
    print(f"   ⚠️  Latency 페널티 발생")
else:
    print(f"   ✅ Latency 페널티 0 (T ≤ 1s)")
print(f"\n📝 정답률은 Kaggle 'Submit to Competition' 후 leaderboard 에서 확인")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  submission.csv saved → /kaggle/working/submission.csv
  rows                  : 180
  md5                   : 96bc30be60a3c80db45b4725f0810ad5
  inference_time_sec    : 1.6123
  override fires        : 5
  veto blocks           : 4
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Final prediction distribution:
    crazing             : 30
    inclusion           : 30
    patches             : 30
    pitted_surface      : 30
    rolled-in_scale     : 30
    scratches           : 30

🏆 T_inference : 1.6123s
   Latency 페널티 : -1.53
   ⚠️  Latency 페널티 발생

📝 정답률은 Kaggle 'Submit to Competition' 후 leaderboard 에서 확인


## ✅ 추론 완료

### Kaggle 제출 시 확인사항

1. **Dataset attach 확인**: 노트북 우측 Input 패널에 Dataset 이 추가됐는지
2. **Internet OFF**: 노트북 설정에서 `Internet: Off` 확인 (대회 룰)
3. **Accelerator: None**: 노트북 설정에서 `Accelerator: None` 확인 (대회 룰)
4. **Output**: `submission.csv` 가 `/kaggle/working/` 에 생성됐는지

### Cross-environment 재현 검증

다른 Kaggle 세션 또는 local 에서 동일 노트북 실행 → 같은 `submission.csv md5` 면 ✅ bit-exact 재현 성공.